# Exercice 11 - Agregations Gold

## Objectifs
- Comprendre la couche Gold du Data Lake
- Creer des agregations metier
- Construire des tables de faits et dimensions
- Optimiser pour la BI et le reporting

---

## 1. La couche Gold

```
+------------------------------------------------------------------+
|                       COUCHE GOLD                                |
+------------------------------------------------------------------+
|                                                                  |
|  Objectif : Donnees pretes pour la BI et les decisions          |
|                                                                  |
|  +------------------------+   +------------------------+        |
|  | Tables agregees        |   | KPIs et metriques      |        |
|  +------------------------+   +------------------------+        |
|  | - Ventes par jour      |   | - CA total             |        |
|  | - Ventes par produit   |   | - Panier moyen         |        |
|  | - Ventes par region    |   | - Taux de croissance   |        |
|  +------------------------+   +------------------------+        |
|                                                                  |
|  +------------------------+   +------------------------+        |
|  | Dimensions             |   | Tables de faits        |        |
|  +------------------------+   +------------------------+        |
|  | - dim_clients          |   | - fact_ventes          |        |
|  | - dim_produits         |   | - fact_commandes       |        |
|  | - dim_temps            |   | - fact_livraisons      |        |
|  +------------------------+   +------------------------+        |
|                                                                  |
+------------------------------------------------------------------+
```

## 2. Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

# Creer la SparkSession
spark = SparkSession.builder \
    .appName("Agregations Gold") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0,org.apache.hadoop:hadoop-aws:3.4.1,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

date_traitement = datetime.now().strftime("%Y-%m-%d")
print(f"Spark pret - Date : {date_traitement}")

In [ ]:
# Charger les donnees depuis PostgreSQL
jdbc_url = "jdbc:postgresql://postgres:5432/app"
jdbc_properties = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

df_orders = spark.read.jdbc(url=jdbc_url, table="orders", properties=jdbc_properties)
df_order_details = spark.read.jdbc(url=jdbc_url, table="order_details", properties=jdbc_properties)
df_products = spark.read.jdbc(url=jdbc_url, table="products", properties=jdbc_properties)
df_customers = spark.read.jdbc(url=jdbc_url, table="customers", properties=jdbc_properties)
df_categories = spark.read.jdbc(url=jdbc_url, table="categories", properties=jdbc_properties)

print("Donnees chargees")

In [ ]:
# Preparer les donnees de base
df_details = df_order_details.withColumn(
    "montant",
    F.round(F.col("unit_price") * F.col("quantity") * (1 - F.col("discount")), 2)
)

# Joindre avec orders pour avoir la date
df_ventes = df_details.join(
    df_orders.select("order_id", "customer_id", "order_date", "employee_id"),
    on="order_id",
    how="left"
)

print(f"Donnees de ventes preparees : {df_ventes.count()} lignes")

## 3. Agregation par date

In [ ]:
# Ventes par jour
gold_ventes_jour = df_ventes.groupBy("order_date").agg(
    F.countDistinct("order_id").alias("nb_commandes"),
    F.sum("quantity").alias("nb_articles"),
    F.round(F.sum("montant"), 2).alias("ca_total"),
    F.round(F.avg("montant"), 2).alias("panier_moyen_ligne")
).orderBy("order_date")

print("Ventes par jour :")
gold_ventes_jour.show(10)

In [ ]:
# Ventes par mois
gold_ventes_mois = df_ventes \
    .withColumn("annee", F.year("order_date")) \
    .withColumn("mois", F.month("order_date")) \
    .groupBy("annee", "mois").agg(
        F.countDistinct("order_id").alias("nb_commandes"),
        F.round(F.sum("montant"), 2).alias("ca_total")
    ).orderBy("annee", "mois")

print("Ventes par mois :")
gold_ventes_mois.show()

In [ ]:
# Ajouter le taux de croissance mois sur mois
window_mois = Window.orderBy("annee", "mois")

gold_ventes_mois_growth = gold_ventes_mois.withColumn(
    "ca_mois_precedent",
    F.lag("ca_total", 1).over(window_mois)
).withColumn(
    "croissance_pct",
    F.round((F.col("ca_total") - F.col("ca_mois_precedent")) / F.col("ca_mois_precedent") * 100, 2)
)

gold_ventes_mois_growth.show()

## 4. Agregation par produit

In [ ]:
# Enrichir avec les noms de produits
df_ventes_produit = df_ventes.join(
    df_products.select("product_id", "product_name", "category_id"),
    on="product_id",
    how="left"
).join(
    df_categories.select("category_id", "category_name"),
    on="category_id",
    how="left"
)

In [ ]:
# Top produits par CA
gold_top_produits = df_ventes_produit.groupBy("product_id", "product_name").agg(
    F.sum("quantity").alias("quantite_vendue"),
    F.round(F.sum("montant"), 2).alias("ca_total"),
    F.countDistinct("order_id").alias("nb_commandes")
).orderBy(F.desc("ca_total"))

print("Top 10 produits par CA :")
gold_top_produits.show(10)

In [ ]:
# Ventes par categorie
gold_categories = df_ventes_produit.groupBy("category_id", "category_name").agg(
    F.countDistinct("product_id").alias("nb_produits"),
    F.sum("quantity").alias("quantite_vendue"),
    F.round(F.sum("montant"), 2).alias("ca_total")
).orderBy(F.desc("ca_total"))

print("Ventes par categorie :")
gold_categories.show()

## 5. Agregation par client

In [ ]:
# Enrichir avec les clients
df_ventes_client = df_ventes.join(
    df_customers.select("customer_id", "company_name", "country"),
    on="customer_id",
    how="left"
)

In [ ]:
# Analyse RFM (Recence, Frequence, Montant)
date_reference = df_ventes_client.agg(F.max("order_date")).collect()[0][0]

gold_rfm = df_ventes_client.groupBy("customer_id", "company_name", "country").agg(
    F.datediff(F.lit(date_reference), F.max("order_date")).alias("recence_jours"),
    F.countDistinct("order_id").alias("frequence"),
    F.round(F.sum("montant"), 2).alias("montant_total")
)

print("Analyse RFM :")
gold_rfm.orderBy(F.desc("montant_total")).show(10)

In [ ]:
# Segmentation clients
gold_segments = gold_rfm.withColumn(
    "segment",
    F.when((F.col("recence_jours") <= 30) & (F.col("frequence") >= 5), "VIP")
     .when((F.col("recence_jours") <= 60) & (F.col("frequence") >= 3), "Fidele")
     .when(F.col("recence_jours") <= 90, "Actif")
     .when(F.col("recence_jours") <= 180, "A risque")
     .otherwise("Inactif")
)

# Repartition par segment
gold_segments.groupBy("segment").agg(
    F.count("*").alias("nb_clients"),
    F.round(F.sum("montant_total"), 2).alias("ca_total")
).orderBy(F.desc("ca_total")).show()

## 6. Agregation geographique

In [ ]:
# Ventes par pays
gold_pays = df_ventes_client.groupBy("country").agg(
    F.countDistinct("customer_id").alias("nb_clients"),
    F.countDistinct("order_id").alias("nb_commandes"),
    F.round(F.sum("montant"), 2).alias("ca_total")
).withColumn(
    "ca_moyen_client",
    F.round(F.col("ca_total") / F.col("nb_clients"), 2)
).orderBy(F.desc("ca_total"))

print("Ventes par pays :")
gold_pays.show()

## 7. KPIs globaux

In [ ]:
# Calculer les KPIs
kpis = df_ventes.agg(
    F.round(F.sum("montant"), 2).alias("ca_total"),
    F.countDistinct("order_id").alias("nb_commandes_total"),
    F.countDistinct("customer_id").alias("nb_clients_actifs"),
    F.sum("quantity").alias("nb_articles_vendus")
).collect()[0]

print("=" * 50)
print("KPIs GLOBAUX")
print("=" * 50)
print(f"CA Total          : {kpis['ca_total']:,.2f} EUR")
print(f"Nb Commandes      : {kpis['nb_commandes_total']:,}")
print(f"Nb Clients Actifs : {kpis['nb_clients_actifs']:,}")
print(f"Nb Articles Vendus: {kpis['nb_articles_vendus']:,}")
print(f"Panier Moyen      : {kpis['ca_total'] / kpis['nb_commandes_total']:,.2f} EUR")
print("=" * 50)

In [ ]:
# Creer une table de KPIs
gold_kpis = spark.createDataFrame([
    ("ca_total", float(kpis['ca_total']), "EUR"),
    ("nb_commandes", float(kpis['nb_commandes_total']), "count"),
    ("nb_clients", float(kpis['nb_clients_actifs']), "count"),
    ("nb_articles", float(kpis['nb_articles_vendus']), "count"),
    ("panier_moyen", float(kpis['ca_total'] / kpis['nb_commandes_total']), "EUR")
], ["kpi", "valeur", "unite"])

gold_kpis = gold_kpis.withColumn("date_calcul", F.lit(date_traitement))
gold_kpis.show()

## 8. Sauvegarder les tables Gold

In [ ]:
# Sauvegarder toutes les tables Gold
tables_gold = {
    "ventes_jour": gold_ventes_jour,
    "ventes_mois": gold_ventes_mois_growth,
    "top_produits": gold_top_produits,
    "categories": gold_categories,
    "clients_rfm": gold_segments,
    "ventes_pays": gold_pays,
    "kpis": gold_kpis
}

print("Sauvegarde des tables Gold :")
print("=" * 50)

for nom, df in tables_gold.items():
    chemin = f"s3a://gold/{nom}/{date_traitement}"
    df.write.mode("overwrite").parquet(chemin)
    print(f"[OK] {nom} : {df.count()} lignes -> {chemin}")

print("=" * 50)
print("Sauvegarde terminee")

---

## Exercice

**Objectif** : Creer une analyse Gold des employes

**Consigne** :
1. Calculez les ventes par employe (nb commandes, CA)
2. Classez les employes par performance
3. Calculez la part de CA de chaque employe (en %)
4. Sauvegardez dans Gold

A vous de jouer :

In [ ]:
# TODO: Charger employees
df_employees = spark.read.jdbc(url=jdbc_url, table="employees", properties=jdbc_properties)
df_employees.show(5)

In [ ]:
# TODO: Calculer les ventes par employe

In [ ]:
# TODO: Classer et calculer les parts

---

## Resume

Dans ce notebook, vous avez appris :
- Le role de la couche **Gold** dans un Data Lake
- Comment creer des **agregations temporelles** (jour, mois)
- Comment calculer des **KPIs metier**
- Comment faire une **analyse RFM** client
- Comment **segmenter** les donnees
- Comment **sauvegarder** plusieurs tables Gold

### Prochaine etape
Dans le prochain notebook, nous apprendrons les operations avancees sur les DataFrames.